In [10]:
import s2sphere
import shapely.geometry as geom
from typing import Set
from typing import Iterable 
import geopandas as gpd
import hashlib
import psycopg2

In [ ]:
# ------------------- PostgreSQL Config -------------------
DB_CONFIG = {
    "dbname": "your_db_name",
    "user": "your_db_user",
    "password": "your_password",
    "host": "db_host",
    "port": "port"
}

# ------------------- DB Connection Helper -------------------
def get_connection():
    return psycopg2.connect(**DB_CONFIG)

In [12]:
def cover_polygon_with_s2(poly: geom.Polygon,min_level: int = 10,max_level: int = 20) -> Set[str]:
    """
    Returns a set of S2 cell-tokens that cover `poly`
    at resolutions between min_level (coarsest) and max_level (finest).
    """

    # 1. Cover the polygon's bounding box with level=min_level cells
    coverer = s2sphere.RegionCoverer()
    coverer.min_level = min_level
    coverer.max_level = min_level
    coverer.max_cells = 0  # unlimited for bounding‐box pass

    # build an S2 LatLngRect for the polygon's bbox
    minx, miny, maxx, maxy = poly.bounds
    ll_lo = s2sphere.LatLng.from_degrees(miny, minx)
    ll_hi = s2sphere.LatLng.from_degrees(maxy, maxx)
    bbox = s2sphere.LatLngRect.from_point_pair(ll_lo, ll_hi)

    initial_cells = coverer.get_covering(bbox)

    out_tokens = set()

    def recurse_cell(cell_id: s2sphere.CellId):
        """
        - If the S2 cell is fully inside poly → add & stop  
        - Elif at max_level            → add & stop  
        - Elif intersects poly         → subdivide  
        """
        # get the S2 cell's lat/lng rectangle
        cell = s2sphere.Cell(cell_id)
        rect: s2sphere.LatLngRect = cell.get_rect_bound()
        lo: s2sphere.LatLng = rect.lo()
        hi: s2sphere.LatLng = rect.hi()

        # build a Shapely polygon for this S2 cell
        cell_poly = geom.Polygon([
            (lo.lng().degrees, lo.lat().degrees),
            (lo.lng().degrees, hi.lat().degrees),
            (hi.lng().degrees, hi.lat().degrees),
            (hi.lng().degrees, lo.lat().degrees),
            (lo.lng().degrees, lo.lat().degrees),
        ])

        if poly.contains(cell_poly) or cell_id.level() >= max_level:
            out_tokens.add(cell_id.to_token())
        elif poly.intersects(cell_poly):
            # subdivide into its 4 children
            for child in cell_id.children():
                recurse_cell(child)

    # 2. Recurse on each initial cell
    for cid in initial_cells:
        recurse_cell(cid)

    return out_tokens

In [13]:
def generate_regionID(tokens: Iterable[str]) -> str:
    """
    Generate a deterministic 64-character alphanumeric ID (hex) for a unique set of tokens,
    formatted as eight 8-character chunks separated by '-'.

    Parameters
    ----------
    tokens : Iterable[str]
        An unordered collection of string tokens (e.g. S2 cell tokens).

    Returns
    -------
    str
        A 64-character hex string split into 8-char groups, e.g.
        '3b1f5a9c-2d4e6f7a-8b9c0d1e-2f3a4b5c-6d7e8f9a-0b1c2d3e-4f5a6b7c-8d9e0f1a'
    """
    # 1. Sort tokens so order is irrelevant
    sorted_tokens = sorted(tokens)

    # 2. Build the SHA-256 hash with a clear delimiter
    m = hashlib.sha256()
    for tok in sorted_tokens:
        m.update(b"|")
        m.update(tok.encode("utf-8"))
   
    # 3. Full 64-character hex digest
    full_hex = m.hexdigest()  # length == 64

    # 4. Split into eight 16-character chunks to make these unique
    parts = [full_hex[i:i+16] for i in range(0, 64, 16)]

    # 5. Join with '-'
    return "-".join(parts)

In [14]:
# create regions table 
def create_regions_table():
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        # Create table with region_id as primary key
        create_table_query = """
        CREATE TABLE IF NOT EXISTS regions (
            region_id VARCHAR(255) PRIMARY KEY,
            region_boundary TEXT NOT NULL
        );
        """
        cursor.execute(create_table_query)
        
        conn.commit()
        print("Regions table created successfully")
        
    except Exception as e:
        print(f"Error creating table: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()

In [15]:
# store region_id , region_boundary(poly.wkt) in db
def store_region_data(region_id, region_boundary):
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        # Check if region_id already exists
        check_query = """
        SELECT region_id 
        FROM regions 
        WHERE region_id = %s
        """
        cursor.execute(check_query, (region_id,))
        
        if cursor.fetchone():
            return f"Region with ID {region_id} already exists"
        
        # Insert new region data
        insert_query = """
        INSERT INTO regions (region_id, region_boundary)
        VALUES (%s, %s)
        """
        cursor.execute(insert_query, (region_id, region_boundary))
        conn.commit()
        return f"Successfully stored region {region_id}"
        
    except Exception as e:
        print(f"Error storing region data: {e}")
        conn.rollback()
        return f"Failed to store region: {e}"
    finally:
        cursor.close()
        conn.close()

In [16]:
#read regions from geojson file

regions_gdf = gpd.read_file('custom_aldeas.geojson')
regions_gdf = regions_gdf.set_crs(epsg=4326, allow_override=True)

region_ids = []

def process_and_store_regions():
    
    # Create table first
    create_regions_table()
    
    # Iterate through polygons
    for i, row in regions_gdf.iterrows():
        poly = regions_gdf.geometry.iloc[i]
        tokens = cover_polygon_with_s2(poly, min_level=5, max_level=19)
        region_id = generate_regionID(tokens)
        region_ids.append(region_id)
        print(f"==poly wkt===: {poly.wkt}")
        print(f"Region ID: {region_id}")
        
        # Store in database
        result = store_region_data(region_id, poly.wkt)
        print(f"Storage result: {result}")
        print('\n')
    
    print(f"region_ids : {region_ids}")

In [17]:
process_and_store_regions()

Regions table created successfully
==poly wkt===: POLYGON ((-88.3264 15.4571, -88.3265 15.4586, -88.3263 15.462, -88.3262 15.4647, -88.3261 15.4665, -88.326 15.4684, -88.3256 15.4709, -88.3255 15.4727, -88.3256 15.4746, -88.3255 15.4766, -88.3254 15.4785, -88.3252 15.4783, -88.3237 15.4797, -88.3218 15.4808, -88.3179 15.4823, -88.3132 15.4832, -88.3095 15.4835, -88.3068 15.4828, -88.3048 15.4807, -88.3031 15.478, -88.3009 15.4769, -88.2979 15.4773, -88.2934 15.4756, -88.2907 15.4751, -88.2874 15.4753, -88.2844 15.4753, -88.2826 15.4736, -88.2821 15.4716, -88.2814 15.469, -88.2799 15.4664, -88.2771 15.4646, -88.2748 15.463, -88.2722 15.4608, -88.2704 15.4592, -88.2684 15.4562, -88.2673 15.4532, -88.2662 15.45, -88.264 15.4472, -88.2621 15.4463, -88.2602 15.4459, -88.2595 15.4459, -88.2593 15.4454, -88.2591 15.4451, -88.2586 15.4445, -88.2578 15.4436, -88.2571 15.4428, -88.2571 15.4427, -88.2563 15.4415, -88.2547 15.4389, -88.2545 15.4366, -88.255 15.4346, -88.256 15.4331, -88.2571 15.43